In [1]:
import pandas as pd
import re
from collections import Counter
import numpy as np
from datetime import datetime
import os

**Поиск паттернов**

In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
from datetime import datetime
import os

df = pd.read_excel('C://Users//molot//Downloads//Flash call.xlsx', engine='openpyxl')
phone_numbers = df.iloc[:, 0].astype(str).tolist()

def analyze_by_operator_prefix(phone_numbers_12_digit):
    """
    Анализ группировки по префиксам операторов (первые 4 цифры)
    с проверкой вариативности верификационных кодов
    """
    print("\n=== АНАЛИЗ ПО ПРЕФИКСАМ ОПЕРАТОРОВ ===")
    
    prefix_stats = {}
    
    for number in phone_numbers_12_digit:
        operator_prefix = number[:4]  # первые 4 цифры - код оператора
        verification_code = number[4:10]  # позиции 5-10 - верификационный код
        
        if operator_prefix not in prefix_stats:
            prefix_stats[operator_prefix] = {
                'count': 0,
                'verification_codes': set(),
                'suffixes': set(),  # последние 2 цифры
                'numbers': []
            }
        
        prefix_stats[operator_prefix]['count'] += 1
        prefix_stats[operator_prefix]['verification_codes'].add(verification_code)
        prefix_stats[operator_prefix]['suffixes'].add(number[10:12])
        prefix_stats[operator_prefix]['numbers'].append(number)
    
    # Анализ каждого префикса
    flash_call_candidates = []
    
    for prefix, stats in prefix_stats.items():
        total_numbers = stats['count']
        unique_codes = len(stats['verification_codes'])
        code_variability = unique_codes / total_numbers
        suffix_variability = len(stats['suffixes']) / total_numbers
        
        # Критерии Flash Call:
        is_flash_call = (code_variability > 0.8 and total_numbers >= 10)
        
        flash_call_candidates.append({
            'operator_prefix': prefix,
            'total_numbers': total_numbers,
            'unique_verification_codes': unique_codes,
            'code_variability': code_variability,
            'unique_suffixes': len(stats['suffixes']),
            'suffix_variability': suffix_variability,
            'is_flash_call': is_flash_call,
            'confidence': 'high' if (code_variability > 0.9 and total_numbers >= 20) else 'medium'
        })
    
    # Сортировка по количеству номеров
    flash_call_candidates.sort(key=lambda x: x['total_numbers'], reverse=True)
    return flash_call_candidates

def generate_wildcard_patterns(operator_prefix, suffix_analysis, code_analysis):
    patterns = []
    
    # Базовый паттерн: префикс оператора + любые цифры
    patterns.append({
        'pattern': f"{operator_prefix}*",
        'type': 'broad',
        'specificity': 'low',
        'risk_normal_traffic': 'medium',
        'description': f'Блокировка всех номеров с префиксом {operator_prefix}'
    })
    
    if suffix_analysis and len(suffix_analysis) <= 5:
        suffixes = list(suffix_analysis.keys())
        if len(suffixes) == 1:
            patterns.append({
                'pattern': f"{operator_prefix}??????{suffixes[0]}",
                'type': 'specific_suffix',
                'specificity': 'high', 
                'risk_normal_traffic': 'low',
                'description': f'Префикс {operator_prefix} + 6 любых цифр + суффикс {suffixes[0]}'
            })
        else:
            for suffix in suffixes[:3]:
                patterns.append({
                    'pattern': f"{operator_prefix}??????{suffix}",
                    'type': 'specific_suffix',
                    'specificity': 'high',
                    'risk_normal_traffic': 'low',
                    'description': f'Префикс {operator_prefix} + 6 любых цифр + суффикс {suffix}'
                })
    
    patterns.append({
        'pattern': f"{operator_prefix}??????**",
        'type': 'fixed_length',
        'specificity': 'medium',
        'risk_normal_traffic': 'low',
        'description': f'Префикс {operator_prefix} + 6 любых цифр кода + 2 любые цифры суффикса'
    })
    
    return patterns

def analyze_suffix_distribution(phone_numbers_12_digit, operator_prefix):
    suffixes = [num[10:12] for num in phone_numbers_12_digit if num.startswith(operator_prefix)]
    return Counter(suffixes)

def analyze_code_patterns(phone_numbers_12_digit, operator_prefix):
    codes = [num[4:10] for num in phone_numbers_12_digit if num.startswith(operator_prefix)]
    code_analysis = {
        'total_codes': len(codes),
        'unique_codes': len(set(codes)),
        'variability': len(set(codes)) / len(codes) if codes else 0,
        'most_common_codes': Counter(codes).most_common(3)
    }
    
    return code_analysis

def calculate_blocking_risk(operator_prefix, total_numbers, code_variability, suffix_variability):
    risk_score = 0
    
    if total_numbers > 100:
        risk_score += 2
    elif total_numbers > 50:
        risk_score += 1
    
    if code_variability > 0.9:
        risk_score -= 2
    elif code_variability > 0.8:
        risk_score -= 1

    if suffix_variability < 0.3:
        risk_score -= 1
    
    if risk_score <= -2:
        return 'low'
    elif risk_score <= 0:
        return 'medium'
    else:
        return 'high'

def generate_blocking_recommendations_with_wildcards(flash_call_candidates, phone_numbers_12_digit):
    blocking_patterns = []
    
    for candidate in flash_call_candidates:
        if candidate['is_flash_call']:
            operator_prefix = candidate['operator_prefix']
            suffix_distribution = analyze_suffix_distribution(phone_numbers_12_digit, operator_prefix)
            code_analysis = analyze_code_patterns(phone_numbers_12_digit, operator_prefix)
            
            blocking_risk = calculate_blocking_risk(
                operator_prefix,
                candidate['total_numbers'],
                candidate['code_variability'],
                candidate['suffix_variability']
            )
            
            wildcard_patterns = generate_wildcard_patterns(
                operator_prefix, 
                suffix_distribution, 
                code_analysis
            )
            for pattern in wildcard_patterns:
                pattern.update({
                    'operator_prefix': operator_prefix,
                    'coverage': candidate['total_numbers'],
                    'code_variability': candidate['code_variability'],
                    'suffix_variability': candidate['suffix_variability'],
                    'blocking_risk': blocking_risk,
                    'confidence': candidate['confidence']
                })
            
            blocking_patterns.extend(wildcard_patterns)
    
    specificity_order = {'high': 3, 'medium': 2, 'low': 1}
    blocking_patterns.sort(key=lambda x: (
        specificity_order[x['specificity']], 
        x['coverage']
    ), reverse=True)
    
    final_patterns = []
    seen_operators = set()
    
    for pattern in blocking_patterns:
        operator = pattern['operator_prefix']
        if operator not in seen_operators:
            seen_operators.add(operator)
            final_patterns.append(pattern)
        elif pattern['specificity'] == 'high' and pattern['blocking_risk'] == 'low':
            for i, existing in enumerate(final_patterns):
                if existing['operator_prefix'] == operator and existing['specificity'] != 'high':
                    final_patterns[i] = pattern
                    break
    
    print("Паттерн              | Тип             | Специфичность | Риск   | Номеров | Описание")
    print("-" * 110)
    
    for pattern in final_patterns[:25]:
        print(f"{pattern['pattern']:20} | {pattern['type']:13} | {pattern['specificity']:13} | {pattern['blocking_risk']:6} | {pattern['coverage']:7} | {pattern['description']}")
    
    return pd.DataFrame(final_patterns)

def save_blocking_patterns_for_system(blocking_patterns_df):
    timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
    results_dir = f'blocking_system_patterns_{timestamp}'
    
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    
    system_file = f'{results_dir}/blocking_patterns_system_{timestamp}.csv'
    
    system_data = []
    for _, pattern in blocking_patterns_df.iterrows():
        system_data.append({
            'pattern': pattern['pattern'],
            'priority': 'HIGH' if pattern['confidence'] == 'high' else 'MEDIUM',
            'risk_level': pattern['blocking_risk'].upper(),
            'expected_coverage': pattern['coverage'],
            'description': pattern['description']
        })
    
    system_df = pd.DataFrame(system_data)
    system_df.to_csv(system_file, index=False, encoding='utf-8')
    
    detail_file = f'{results_dir}/blocking_patterns_detailed_{timestamp}.xlsx'
    
    with pd.ExcelWriter(detail_file, engine='openpyxl') as writer:
        blocking_patterns_df.to_excel(writer, sheet_name='Все паттерны', index=False)
        
        high_priority = blocking_patterns_df[
            (blocking_patterns_df['confidence'] == 'high') & 
            (blocking_patterns_df['blocking_risk'] == 'low')
        ]
        if not high_priority.empty:
            high_priority.to_excel(writer, sheet_name='Высокий приоритет', index=False)
        
        immediate_block = blocking_patterns_df[
            (blocking_patterns_df['specificity'] == 'high') & 
            (blocking_patterns_df['blocking_risk'] == 'low') &
            (blocking_patterns_df['coverage'] >= 20)
        ]
        if not immediate_block.empty:
            immediate_block.to_excel(writer, sheet_name='Немедленная блокировка', index=False)

    generate_system_commands(blocking_patterns_df, results_dir, timestamp)   
    return system_file, detail_file

def generate_system_commands(blocking_patterns_df, results_dir, timestamp):
    commands_file = f'{results_dir}/system_commands_example_{timestamp}.txt'
    
    with open(commands_file, 'w', encoding='utf-8') as f:
        f.write("ВЫСОКИЙ ПРИОРИТЕТ (низкий риск, высокая специфичность):\n")
        high_priority = blocking_patterns_df[
            (blocking_patterns_df['specificity'] == 'high') & 
            (blocking_patterns_df['blocking_risk'] == 'low')
        ]
        
        for _, pattern in high_priority.head(10).iterrows():
            f.write(f"{pattern['pattern']} PRIORITY=HIGH RISK=LOW\n")
            f.write(f"{pattern['description']}\n")
            f.write(f"{pattern['coverage']} номеров\n\n")
        
        f.write("\nСРЕДНИЙ ПРИОРИТЕТ:\n")
        medium_priority = blocking_patterns_df[
            (blocking_patterns_df['specificity'] == 'medium') & 
            (blocking_patterns_df['blocking_risk'] == 'low')
        ]
        
        for _, pattern in medium_priority.head(10).iterrows():
            f.write(f"{pattern['pattern']} PRIORITY=MEDIUM RISK=LOW\n")
            f.write(f"# {pattern['description']}\n\n")

# ОСНОВНОЙ АНАЛИЗ
print(f"Всего номеров: {len(phone_numbers)}")
all_results = {}

try:
    numbers_12_digit = [num for num in phone_numbers if len(num) == 12]
    print(f"12-значных номеров: {len(numbers_12_digit)}/{len(phone_numbers)}")
    
    if numbers_12_digit:
        # ДОБАВЛЕНО: анализ префиксов операторов
        flash_call_candidates = analyze_by_operator_prefix(numbers_12_digit)
        
        blocking_patterns = generate_blocking_recommendations_with_wildcards(
            flash_call_candidates, 
            numbers_12_digit
        )
        all_results['blocking_patterns'] = blocking_patterns
        
        system_file, detail_file = save_blocking_patterns_for_system(blocking_patterns)
        
        total_flash_call_numbers = sum(candidate['total_numbers'] for candidate in flash_call_candidates 
                                     if candidate['is_flash_call'])
        
        high_specificity = len(blocking_patterns[blocking_patterns['specificity'] == 'high'])
        low_risk = len(blocking_patterns[blocking_patterns['blocking_risk'] == 'low'])
        
        print(f"Номеров, покрываемых блокировкой: {total_flash_call_numbers}")
        print(f"Сгенерировано паттернов: {len(blocking_patterns)}")
        print(f"Высокоспецифичных паттернов: {high_specificity}")
        print(f"Паттернов с низким риском: {low_risk}")
        print(f"\nФайлы сохранены в папке: blocking_system_patterns_...")

except Exception as e:
    print(f"Ошибка при анализе: {e}")
    import traceback
    traceback.print_exc()

c:\Users\molot\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Всего номеров: 4722
12-значных номеров: 4722/4722

=== АНАЛИЗ ПО ПРЕФИКСАМ ОПЕРАТОРОВ ===
Топ префиксов операторов:
Префикс | Номеров | Вариативность кодов | Flash Call
------------------------------------------------------------
9929    |      80 |             100.0% | ✓
3512    |      48 |             100.0% | ✓
3327    |      45 |             100.0% | ✓
3358    |      43 |             100.0% | ✓
3374    |      42 |             100.0% | ✓
4455    |      42 |             100.0% | ✓
4448    |      41 |             100.0% | ✓
3344    |      40 |             100.0% | ✓
3365    |      40 |             100.0% | ✓
3324    |      39 |             100.0% | ✓
3376    |      39 |             100.0% | ✓
4927    |      39 |             100.0% | ✓
3364    |      38 |             100.0% | ✓
3367    |      38 |             100.0% | ✓
4928    |      38 |             100.0% | ✓
Паттерн              | Тип             | Специфичность | Риск   | Номеров | Описание
----------------------------------------